# 👥 Agent Teams: Multi-Agent Architectures & Collaboration

## Introduction to Agent Teams

**Agent Teams** enable building sophisticated multi-agent systems where multiple specialized agents collaborate to solve complex problems. Instead of a single agent handling all tasks, an agent team decomposes problems into sub-tasks, with each agent specializing in specific roles and responsibilities.

### Why Use Agent Teams?

**Single Agent Limitations:**
- ❌ Cannot parallelize independent subtasks
- ❌ Limited specialization (one agent, one knowledge base)
- ❌ Poor at handling truly complex, multi-domain problems
- ❌ Difficult to maintain separation of concerns

**Agent Teams Benefits:**
- ✅ **Parallelization** — Multiple agents work concurrently on different tasks
- ✅ **Specialization** — Each agent has specific expertise and tools
- ✅ **Scalability** — Add more agents for larger problems
- ✅ **Maintainability** — Clear roles and responsibilities
- ✅ **Resilience** — If one agent fails, others continue
- ✅ **Real-World Simulation** — Models how humans solve complex problems


## Setup & Dependencies


In [ ]:
import os
from dotenv import load_dotenv
import asyncio
from typing import Any, Dict, List

# Load environment variables
load_dotenv()

# Ensure API keys are set
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["MISTRAL_API_KEY"] = os.getenv("MISTRAL_API_KEY")

print("✅ Dependencies loaded successfully")

## Part 1: Agent Team Architectures

### Architecture 1: Orchestrator Pattern

One agent coordinates and delegates tasks to specialized agents.

```
┌─────────────────────────┐
│  Orchestrator Agent     │
│  (Task Delegation)      │
└───────────┬─────────────┘
            │
    ┌───────┼───────┐
    │       │       │
    ▼       ▼       ▼
┌────────┐ ┌────────┐ ┌────────┐
│Research│ │Analysis│ │Execution│
│ Agent  │ │ Agent  │ │ Agent  │
└────────┘ └────────┘ └────────┘
```

**Use Case:** Customer support, research automation, task management


In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate

# Define specialized tools
@tool
def research_market(topic: str) -> str:
    """Research a market topic and return key insights"""
    research_data = {
        "AI market": "The AI market is growing at 40% CAGR with enterprise adoption increasing",
        "cloud computing": "Cloud services are consolidating around major providers (AWS, Azure, GCP)",
        "cybersecurity": "Zero-trust architecture is becoming industry standard"
    }
    return research_data.get(topic.lower(), f"Research data for {topic} not available")

@tool
def analyze_data(data: str, metric: str) -> str:
    """Analyze data and compute metrics"""
    analysis = {
        "growth": f"Analysis: {data[:50]}... shows {45}% growth potential",
        "risk": f"Analysis: {data[:50]}... indicates moderate risk",
        "roi": f"Analysis: {data[:50]}... projects 3.2x ROI"
    }
    return analysis.get(metric.lower(), "Metric not found")

@tool
def generate_report(analysis_result: str) -> str:
    """Generate a business report based on analysis"""
    return f"📊 Executive Report:\n{analysis_result}\n\nRecommendation: Proceed with investment strategy."

# Define the orchestrator agent
llm = ChatOpenAI(model="gpt-4", temperature=0.7)

orchestrator_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    You are an intelligent orchestrator managing a team of specialized agents.
    Your job is to:
    1. Understand the user's goal
    2. Delegate tasks to appropriate agents (research, analysis, reporting)
    3. Combine results into a coherent response
    
    Available tools:
    - research_market: For gathering market insights
    - analyze_data: For analyzing information
    - generate_report: For creating reports
    """)
,
    ("human", "{input}")
])

orchestrator = create_tool_calling_agent(
    llm,
    [research_market, analyze_data, generate_report],
    orchestrator_prompt
)

orchestrator_executor = AgentExecutor(
    agent=orchestrator,
    tools=[research_market, analyze_data, generate_report],
    verbose=True
)

print("✅ Orchestrator agent created and ready to delegate tasks")

In [ ]:
# Run the orchestrator pattern example
result = orchestrator_executor.invoke({
    "input": "Research the AI market, analyze its growth potential, and generate an investment report"
})

print("\n" + "="*80)
print("ORCHESTRATOR RESULT:")
print("="*80)
print(result["output"])

## Part 2: Multi-Agent Collaboration

### Architecture 2: Peer-to-Peer Agent Teams

Specialized agents work in sequence, each handling their domain of expertise.

```
Market Agent → Sentiment Agent → Report Agent → Final Output
```


In [ ]:
# Define specialized agents for financial analysis

@tool
def fetch_market_data(symbol: str) -> Dict[str, Any]:
    """Fetch current market data for a stock symbol"""
    mock_data = {
        "AAPL": {"price": 150.25, "volume": 52_000_000, "change": 2.5},
        "MSFT": {"price": 380.50, "volume": 18_500_000, "change": 1.8},
        "GOOGL": {"price": 140.75, "volume": 21_200_000, "change": 3.2}
    }
    return mock_data.get(symbol.upper(), {"error": f"Symbol {symbol} not found"})

@tool
def analyze_sentiment(text: str) -> str:
    """Analyze sentiment of market news or data"""
    if any(word in text.lower() for word in ["growth", "increase", "positive"]):
        return "📈 POSITIVE: Market sentiment is bullish"
    elif any(word in text.lower() for word in ["decline", "loss", "negative"]):
        return "📉 NEGATIVE: Market sentiment is bearish"
    else:
        return "➡️ NEUTRAL: Market sentiment is mixed"

@tool
def generate_investment_report(analysis: str, sentiment: str) -> str:
    """Generate an investment recommendation report"""
    return f"""
    📊 INVESTMENT REPORT
    ====================
    Analysis: {analysis}
    Sentiment: {sentiment}
    
    Recommendation: Based on the analysis and sentiment, consider a HOLD position.
    Risk Level: MODERATE
    Time Horizon: 3-6 months
    """

# Create individual specialized agents
market_agent_llm = ChatOpenAI(model="gpt-4", temperature=0)
sentiment_agent_llm = ChatOpenAI(model="gpt-4", temperature=0)
report_agent_llm = ChatOpenAI(model="gpt-4", temperature=0.7)

# Market Agent
market_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a market data specialist. Fetch and analyze market data."),
    ("human", "{input}")
])

market_agent = create_tool_calling_agent(
    market_agent_llm,
    [fetch_market_data],
    market_prompt
)

market_executor = AgentExecutor(agent=market_agent, tools=[fetch_market_data], verbose=False)

# Sentiment Agent
sentiment_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a sentiment analysis specialist. Analyze text and market sentiment."),
    ("human", "{input}")
])

sentiment_agent = create_tool_calling_agent(
    sentiment_agent_llm,
    [analyze_sentiment],
    sentiment_prompt
)

sentiment_executor = AgentExecutor(agent=sentiment_agent, tools=[analyze_sentiment], verbose=False)

# Report Agent
report_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a financial report writer. Generate comprehensive investment reports."),
    ("human", "{input}")
])

report_agent = create_tool_calling_agent(
    report_agent_llm,
    [generate_investment_report],
    report_prompt
)

report_executor = AgentExecutor(agent=report_agent, tools=[generate_investment_report], verbose=False)

print("✅ Multi-agent team created with 3 specialized agents")

In [ ]:
# Run the multi-agent collaboration workflow

print("🚀 Starting Multi-Agent Collaboration...\n")

# Step 1: Market Agent fetches data
print("[Agent 1] Market Specialist: Fetching market data...")
market_result = market_executor.invoke({
    "input": "Get market data for AAPL stock"
})
print(f"Result: {market_result['output']}\n")

# Step 2: Sentiment Agent analyzes
print("[Agent 2] Sentiment Analyst: Analyzing market sentiment...")
sentiment_result = sentiment_executor.invoke({
    "input": f"Analyze this market data: AAPL showing growth with price increase of 2.5%"
})
print(f"Result: {sentiment_result['output']}\n")

# Step 3: Report Agent generates report
print("[Agent 3] Report Writer: Generating investment report...")
report_result = report_executor.invoke({
    "input": f"Generate report combining: Market analysis shows AAPL at $150.25 with positive sentiment"
})
print(f"Result: {report_result['output']}\n")

print("="*80)
print("✅ Multi-Agent Workflow Completed Successfully")
print("="*80)

## Part 3: Agent Types & Responsibilities

### Common Agent Types in Teams

| Agent Type | Role | Example |
|-----------|------|----------|
| **Orchestrator** | Coordinates workflow, delegates tasks | Breaks down user query into subtasks |
| **Researcher** | Gathers and analyzes information | Searches APIs, scrapes data, summarizes |
| **Analyzer** | Processes and interprets data | Runs models, computes statistics |
| **Executor** | Performs actions in the world | Books appointments, sends emails |
| **Reviewer** | Validates results, checks quality | Fact-checks, validates outputs |
| **Memory Agent** | Manages knowledge, context, state | Stores past decisions, manages KGs |
| **Safety Agent** | Monitors and enforces constraints | Prevents harmful actions, audits |


In [ ]:
# Example: Complex Agent Team with Multiple Specialized Roles

@tool
def research_topic(topic: str) -> str:
    """Research a specific topic"""
    return f"Research findings on {topic}: Multiple sources confirm high market demand."

@tool
def validate_data(data: str) -> bool:
    """Validate data quality and accuracy"""
    # Simple validation: check if data is reasonable length
    is_valid = len(data) > 10
    return is_valid

@tool
def store_to_memory(key: str, value: str) -> str:
    """Store information in agent memory"""
    return f"Stored: {key} = {value[:50]}..."

@tool
def check_safety(action: str) -> str:
    """Check if an action is safe to execute"""
    dangerous_keywords = ["delete", "drop", "remove", "dangerous"]
    if any(keyword in action.lower() for keyword in dangerous_keywords):
        return "⚠️ SAFETY CHECK FAILED: Action is potentially harmful"
    return "✅ SAFETY CHECK PASSED: Action is safe to execute"

# Create specialized agents
llm = ChatOpenAI(model="gpt-4", temperature=0.7)

research_agent = create_tool_calling_agent(
    llm,
    [research_topic],
    ChatPromptTemplate.from_messages([
        ("system", "You are a research specialist. Use available tools to gather information."),
        ("human", "{input}")
    ])
)

validator_agent = create_tool_calling_agent(
    llm,
    [validate_data],
    ChatPromptTemplate.from_messages([
        ("system", "You are a data validator. Verify quality and accuracy."),
        ("human", "{input}")
    ])
)

memory_agent = create_tool_calling_agent(
    llm,
    [store_to_memory],
    ChatPromptTemplate.from_messages([
        ("system", "You are a memory manager. Store important information."),
        ("human", "{input}")
    ])
)

safety_agent = create_tool_calling_agent(
    llm,
    [check_safety],
    ChatPromptTemplate.from_messages([
        ("system", "You are a safety monitor. Ensure all actions are safe."),
        ("human", "{input}")
    ])
)

print("✅ Specialized agent team created with 4 different roles")

In [ ]:
# Execute a workflow using the specialized agent team

print("🔄 Running Complex Agent Team Workflow...\n")

# Step 1: Research
print("[1️⃣ Research Agent] Gathering information...")
research_exec = AgentExecutor(agent=research_agent, tools=[research_topic], verbose=False)
research = research_exec.invoke({"input": "Research AI market trends"})
print(f"Output: {research['output']}\n")

# Step 2: Validate
print("[2️⃣ Validator Agent] Checking data quality...")
validator_exec = AgentExecutor(agent=validator_agent, tools=[validate_data], verbose=False)
validation = validator_exec.invoke({"input": f"Validate this data: {research['output']}"})
print(f"Output: {validation['output']}\n")

# Step 3: Store to Memory
print("[3️⃣ Memory Agent] Storing findings...")
memory_exec = AgentExecutor(agent=memory_agent, tools=[store_to_memory], verbose=False)
memory = memory_exec.invoke({"input": "Store the AI market research findings"})
print(f"Output: {memory['output']}\n")

# Step 4: Safety Check
print("[4️⃣ Safety Agent] Checking action safety...")
safety_exec = AgentExecutor(agent=safety_agent, tools=[check_safety], verbose=False)
safety_check = safety_exec.invoke({"input": "Is it safe to publish these findings?"})
print(f"Output: {safety_check['output']}\n")

print("="*80)
print("✅ Complex Agent Team Workflow Completed")
print("="*80)

## Part 4: Key Considerations for Agent Teams

### 1. Coordination Strategy
- How do agents communicate?
- Who decides what task comes next?
- How are conflicts resolved?

### 2. State Management
- Shared knowledge base or isolated?
- How is context passed between agents?
- Who maintains conversation history?

### 3. Tool Allocation
- Which tools belong to which agent?
- Can agents call each other's tools?
- How are permissions managed?

### 4. Failure Handling
- What if one agent fails?
- Should the team retry or escalate?
- How is progress saved?

### 5. Performance Optimization
- Can agents run in parallel?
- How do we minimize API calls?
- When should we cache results?


## Part 5: Real-World Use Cases

### Use Case 1: Customer Support Team
```
User Query
    ↓
Triage Agent (categorizes issue)
    ↓
  ├─→ Knowledge Agent (searches FAQs)
  ├─→ Resolution Agent (solves problem)
  └─→ Escalation Agent (contacts human if needed)
    ↓
Final Response
```

### Use Case 2: Research & Analysis
```
Research Goal
    ↓
  ├─→ Literature Agent (finds papers)
  ├─→ Data Agent (processes datasets)
  ├─→ Writing Agent (drafts report)
  └─→ Review Agent (fact-checks)
    ↓
Final Research Report
```

### Use Case 3: Business Operations
```
Business Need
    ↓
  ├─→ Demand Agent (forecasts sales)
  ├─→ Inventory Agent (manages stock)
  ├─→ Logistics Agent (optimizes shipping)
  └─→ Finance Agent (tracks costs)
    ↓
Optimized Operations
```


## Part 6: Frameworks for Agent Teams

### LangGraph (LangChain's Native Solution)
- Graph-based agent orchestration
- Control flow through nodes and edges
- State management across agents
- Native integration with LangChain

### AutoGen (Microsoft)
- Conversation-based multi-agent framework
- Register custom functions across agents
- Automatic tool use orchestration
- Human-in-the-loop capabilities

### CrewAI
- Role-based agent collaboration
- Built-in memory and delegation
- Task-oriented architecture
- Hierarchical team management


## Summary: Key Takeaways

### What You Learned
1. ✅ **Orchestrator Pattern** — Central coordinator delegating to specialists
2. ✅ **Multi-Agent Collaboration** — Sequential workflow with specialized agents
3. ✅ **Agent Types** — Researchers, Analyzers, Executors, Reviewers, Memory, Safety
4. ✅ **Coordination Strategies** — How agents communicate and coordinate
5. ✅ **Real-World Applications** — Customer support, research, business operations
6. ✅ **Frameworks & Tools** — LangGraph, AutoGen, CrewAI

### Next Steps
- 🔗 Combine Agent Teams with RAG (Notebook 6) for intelligent retrieval
- 📊 Use Middleware (Notebook 7) to monitor team performance
- 🚀 Build your own agent team for your domain
- 📚 Explore LangGraph documentation for advanced orchestration

### Best Practices
1. **Start Simple** — Begin with Orchestrator pattern, evolve to Peer-to-Peer
2. **Clear Roles** — Each agent should have well-defined responsibilities
3. **Error Handling** — Plan for agent failures and recovery
4. **Testing** — Test each agent independently before integration
5. **Monitoring** — Track team performance and agent interactions
6. **Documentation** — Document agent capabilities and limitations

---

**Duration:** ~30 min to run all examples  
**Difficulty:** Intermediate-Advanced  
**Prerequisites:** Notebooks 1-3 (basic agent understanding)
